Étape 1 : Importer les données dans Google Colab

In [1]:
from google.colab import files

# Téléverser le fichier data.csv depuis votre ordinateur
uploaded = files.upload()

Saving data.csv to data.csv


Étape 2 : Chargement, Nettoyage et Injection dans SQLite

In [2]:
import sqlite3
import pandas as pd

# 1. Chargement des données
df = pd.read_csv("data.csv", encoding="ISO-8859-1")

# 2. Nettoyage rapide
df = df.dropna(subset=["CustomerID"])
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

# 3. Préparation des colonnes pour le SQL
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Month"] = df["InvoiceDate"].dt.to_period("M").astype(str)
df["Hour"] = df["InvoiceDate"].dt.hour
df["InvoiceDate"] = df["InvoiceDate"].astype(str)

# 4. Connexion SQLite et création de la table 'sales'
conn = sqlite3.connect("analysis.db")
df.to_sql("sales", conn, if_exists="replace", index=False)

print("✅ Base de données SQLite 'analysis.db' créée avec succès !")

✅ Base de données SQLite 'analysis.db' créée avec succès !


Exécution des 5 requêtes SQL (E-commerce)

1. Les 10 pays générant le plus de chiffre d'affaires

In [3]:
query_1 = """
SELECT
    Country,
    ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
    COUNT(DISTINCT InvoiceNo) AS TotalOrders
FROM sales
GROUP BY Country
ORDER BY TotalRevenue DESC
LIMIT 10;
"""
df_top_countries = pd.read_sql(query_1, conn)
display(df_top_countries)

,Country,TotalRevenue,TotalOrders
0,United Kingdom,7308391.55,16646
1,Netherlands,285446.34,94
2,EIRE,265545.90,260
3,Germany,228867.14,457
4,France,209024.05,389
5,Australia,138521.31,57
6,Spain,61577.11,90
7,Switzerland,56443.95,51
8,Belgium,41196.34,98
9,Sweden,38378.33,36


2. Heures de pointe des achats

In [4]:
query_2 = """
SELECT
    Hour,
    COUNT(DISTINCT InvoiceNo) AS OrderCount,
    ROUND(SUM(TotalPrice), 2) AS TotalRevenue
FROM sales
GROUP BY Hour
ORDER BY OrderCount DESC;
"""
df_peak_hours = pd.read_sql(query_2, conn)
display(df_peak_hours)

,Hour,OrderCount,TotalRevenue
0,12,3130,1378571.48
1,13,2636,1173264.75
2,11,2277,1104558.75
3,14,2274,995629.37
4,10,2226,1261192.57
5,15,2037,966191.75
6,9,1393,842605.17
7,16,1100,468885.80
8,8,555,282115.63
9,17,544,234413.52


3. Clients récurrents vs Clients ponctuels

In [5]:
query_3 = """
WITH CustomerOrders AS (
    SELECT CustomerID, COUNT(DISTINCT InvoiceNo) AS TotalOrders
    FROM sales
    GROUP BY CustomerID
)
SELECT
    CASE
        WHEN TotalOrders > 1 THEN 'Client Récurrent'
        ELSE 'Client Ponctuel'
    END AS CustomerType,
    COUNT(CustomerID) AS CustomerCount,
    ROUND(COUNT(CustomerID) * 100.0 / (SELECT COUNT(*) FROM CustomerOrders), 2) AS Percentage
FROM CustomerOrders
GROUP BY CustomerType;
"""
df_customer_types = pd.read_sql(query_3, conn)
display(df_customer_types)

,CustomerType,CustomerCount,Percentage
0,Client Ponctuel,1493,34.42
1,Client Récurrent,2845,65.58


4. Taille moyenne du panier (Valeur et nombre d'articles par commande)

In [6]:
query_4 = """
WITH OrderTotals AS (
    SELECT
        InvoiceNo,
        SUM(TotalPrice) AS OrderValue,
        SUM(Quantity) AS TotalItems
    FROM sales
    GROUP BY InvoiceNo
)
SELECT
    ROUND(AVG(OrderValue), 2) AS AverageBasketValue,
    ROUND(AVG(TotalItems), 2) AS AverageItemsPerBasket
FROM OrderTotals;
"""
df_avg_basket = pd.read_sql(query_4, conn)
display(df_avg_basket)

,AverageBasketValue,AverageItemsPerBasket
0,480.87,278.86


5. Revenus par mois (Tendance mensuelle)

In [7]:
query_5 = """
SELECT
    Month,
    ROUND(SUM(TotalPrice), 2) AS MonthlyRevenue,
    COUNT(DISTINCT InvoiceNo) AS MonthlyOrders
FROM sales
GROUP BY Month
ORDER BY Month ASC;
"""
df_monthly_revenue = pd.read_sql(query_5, conn)
display(df_monthly_revenue)

,Month,MonthlyRevenue,MonthlyOrders
0,2010-12,572713.89,1400
1,2011-01,569445.04,987
2,2011-02,447137.35,997
3,2011-03,595500.76,1321
4,2011-04,469200.36,1149
5,2011-05,678594.56,1555
6,2011-06,661213.69,1393
7,2011-07,600091.01,1331
8,2011-08,645343.90,1280
9,2011-09,952838.38,1755


Étape 4 : Fermeture de la connexion SQL

In [8]:
conn.close()